# Uniform Cost Search: Trailer from Nairobi to Kisumu

**Search strategy:** Uniform Cost Search (UCS)
**Start:** Nairobi  
**Goal:** Kisumu

This notebook models the highway network as a directed, weighted graph. Each edge cost is the path cost from Part 1 (for example, distance in kilometres). UCS expands the frontier node with the lowest accumulated cost and returns the least-cost route when the goal is removed from the priority queue.

## Problem formulation

- **States:** towns/cities in the highway network.
- **Initial state:** Nairobi.
- **Goal test:** current city is Kisumu.
- **Actions:** travel along one outgoing directed road edge.
- **Step cost:** the Part 1 cost written on that edge.
- **Path cost:** sum of step costs along the route.

UCS is appropriate when edge costs are non-negative and the objective is to minimize total path cost. The direction of an edge matters: an edge from A to B does not imply a return edge from B to A.

In [ ]:
# Illustrative directed graph. Replace these sample edges/costs with Part 1 data.
# Format: (origin, destination, cost)
edges = [
    ('Nairobi', 'Naivasha', 90),
    ('Nairobi', 'Nakuru', 160),
    ('Naivasha', 'Nakuru', 70),
    ('Naivasha', 'Kericho', 190),
    ('Nakuru', 'Kisumu', 175),
    ('Nakuru', 'Kericho', 120),
    ('Kericho', 'Kisumu', 60),
]

start = 'Nairobi'
goal = 'Kisumu'

# Convert the edge list into an adjacency list.
graph = {}
for origin, destination, cost in edges:
    if cost < 0:
        raise ValueError('UCS requires non-negative edge costs.')
    graph.setdefault(origin, []).append((destination, cost))
    graph.setdefault(destination, [])

print(f'{len(graph)} cities and {len(edges)} directed road edges loaded.')
print('Outgoing roads from Nairobi:', graph[start])

## Uniform Cost Search

The priority queue (frontier) is ordered by accumulated path cost. A city is finalized when it is removed from the queue at its lowest known cost. The `best_cost` map prevents a more expensive route to the same city from replacing a cheaper one, and `parent` is used to reconstruct the route. A counter breaks ties consistently when two paths have equal cost.

In [ ]:
from heapq import heappush, heappop
from itertools import count

def uniform_cost_search(graph, start, goal):
    # Entries are (accumulated cost, tie-breaker, city).
    sequence = count()
    frontier = [(0, next(sequence), start)]
    best_cost = {start: 0}
    parent = {start: None}
    expanded = []

    while frontier:
        cost_so_far, _, city = heappop(frontier)

        # Ignore an outdated queue entry if a cheaper route was found later.
        if cost_so_far != best_cost.get(city):
            continue

        expanded.append((city, cost_so_far))
        if city == goal:
            path = []
            current = goal
            while current is not None:
                path.append(current)
                current = parent[current]
            path.reverse()
            return path, cost_so_far, expanded

        for neighbor, step_cost in graph.get(city, []):
            new_cost = cost_so_far + step_cost
            if new_cost < best_cost.get(neighbor, float('inf')):
                best_cost[neighbor] = new_cost
                parent[neighbor] = city
                heappush(frontier, (new_cost, next(sequence), neighbor))

    return None, float('inf'), expanded

path, total_cost, expansion_order = uniform_cost_search(graph, start, goal)

if path is None:
    print(f'No route found from {start} to {goal}.')
else:
    print('UCS expansion order (city, accumulated cost):')
    for city, cost in expansion_order:
        print(f'  {city}: {cost}')
    print('\nOptimal path:', ' -> '.join(path))
    print(f'Total path cost: {total_cost}')

## Result and interpretation

With the illustrative costs above, UCS returns **Nairobi → Nakuru → Kisumu**, with a total cost of **335 cost units** (160 + 175). The search expands Nairobi (0), Naivasha (90), Nakuru (160), Kericho (280), and then Kisumu (335).

## Conclusion

UCS explores candidate routes in increasing order of accumulated cost and returns a least-cost path when all road costs are non-negative. The final route and cost depend on the Part 1 graph entered in the data cell.